<a href="https://colab.research.google.com/github/AmanVerma1110/LLM_Projects-LLM/blob/main/AI%20Agents%20/%20First_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q google-generativeai

In [ ]:
# ── Cell 2: Import & Configure ────────────────────────────────────────────────
import google.generativeai as genai
import time
from google.colab import userdata
from IPython.display import Markdown, display

# Load API key from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    print("✅ API key loaded successfully!")
except Exception as e:
    api_key = None
    print(f"❌ Error loading API key: {e}")
    print("👉 Go to the 🔑 Secrets tab in the left sidebar and add GEMINI_API_KEY")

# Configure Gemini
if api_key:
    genai.configure(api_key=api_key)
    print("✅ Gemini configured and ready!")

✅ API key loaded successfully!
✅ Gemini configured and ready!


In [ ]:
# ── Cell 3a: List all available models ────────────────────────────────────────
print("📋 Available Gemini models that support generateContent:\n")

available_models = []

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        available_models.append(m.name)
        print(f"  ✅ {m.name}")

print(f"\n🔢 Total: {len(available_models)} models found")

📋 Available Gemini models that support generateContent:

  ✅ models/gemini-2.5-flash
  ✅ models/gemini-2.5-pro
  ✅ models/gemini-2.0-flash
  ✅ models/gemini-2.0-flash-001
  ✅ models/gemini-2.0-flash-lite-001
  ✅ models/gemini-2.0-flash-lite
  ✅ models/gemini-2.5-flash-preview-tts
  ✅ models/gemini-2.5-pro-preview-tts
  ✅ models/gemma-4-26b-a4b-it
  ✅ models/gemma-4-31b-it
  ✅ models/gemini-flash-latest
  ✅ models/gemini-flash-lite-latest
  ✅ models/gemini-pro-latest
  ✅ models/gemini-2.5-flash-lite
  ✅ models/gemini-2.5-flash-image
  ✅ models/gemini-3-pro-preview
  ✅ models/gemini-3-flash-preview
  ✅ models/gemini-3.1-pro-preview
  ✅ models/gemini-3.1-pro-preview-customtools
  ✅ models/gemini-3.1-flash-lite-preview
  ✅ models/gemini-3.1-flash-lite
  ✅ models/gemini-3-pro-image-preview
  ✅ models/nano-banana-pro-preview
  ✅ models/gemini-3.1-flash-image-preview
  ✅ models/lyria-3-clip-preview
  ✅ models/lyria-3-pro-preview
  ✅ models/gemini-3.1-flash-tts-preview
  ✅ models/gemini-roboti

In [ ]:
# ── Cell 3b: Chat with Gemini (auto-detect models) ────────────────────────────
import time
from IPython.display import Markdown, display

# ── Step 1: Get user input ─────────────────────────────────────────────────────
message = input("💬 Enter your message to Gemini: ").strip()

if not message:
    message = "Hello, Gemini! This is my first ever message to you!"
    print(f"⚠️  No input. Using default: '{message}'")

print(f"\n📤 Sending: {message}\n" + "─"*50)

# ── Step 2: Retry function ─────────────────────────────────────────────────────
def generate_with_retry(model, message, retries=3, wait=40):
    for attempt in range(1, retries + 1):
        try:
            return model.generate_content(message)
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                if attempt < retries:
                    print(f"⏳ Rate limited. Waiting {wait}s... (attempt {attempt}/{retries})")
                    time.sleep(wait)
                else:
                    raise e
            else:
                raise e

# ── Step 3: Use available models fetched from Cell 3a ─────────────────────────
# Priority order — prefers flash/lite (higher free quota)
PREFERRED_KEYWORDS = ["flash-lite", "flash", "pro-exp", "pro"]

def sort_models(models, keywords):
    ordered = []
    for kw in keywords:
        ordered += [m for m in models if kw in m and m not in ordered]
    ordered += [m for m in models if m not in ordered]
    return ordered

sorted_models = sort_models(available_models, PREFERRED_KEYWORDS)

print("🔍 Will try models in this order:")
for m in sorted_models:
    print(f"   → {m}")
print()

# ── Step 4: Try each model ─────────────────────────────────────────────────────
response = None
used_model = None

for model_name in sorted_models:
    try:
        print(f"⏳ Trying: {model_name}...")
        model = genai.GenerativeModel(model_name=model_name)
        response = generate_with_retry(model, message)
        used_model = model_name
        print(f"✅ Success with: {model_name}")
        break
    except Exception as e:
        if "429" in str(e) or "quota" in str(e).lower():
            print(f"⚠️  Quota exceeded — trying next...\n")
            time.sleep(2)
        elif "404" in str(e):
            print(f"⚠️  Model not found — trying next...\n")
        else:
            print(f"❌ Error: {e}")
            break

# ── Step 5: Display response ───────────────────────────────────────────────────
if response:
    print("\n" + "─"*50)
    display(Markdown(f"**🤖 Gemini ({used_model}) says:**\n\n{response.text}"))
else:
    print("\n❌ All models failed.")
    print("💡 Re-run Cell 3a to refresh the model list and try again.")

💬 Enter your message to Gemini: hi gemini

📤 Sending: hi gemini
──────────────────────────────────────────────────
🔍 Will try models in this order:
   → models/gemini-2.0-flash-lite-001
   → models/gemini-2.0-flash-lite
   → models/gemini-flash-lite-latest
   → models/gemini-2.5-flash-lite
   → models/gemini-3.1-flash-lite-preview
   → models/gemini-3.1-flash-lite
   → models/gemini-2.5-flash
   → models/gemini-2.0-flash
   → models/gemini-2.0-flash-001
   → models/gemini-2.5-flash-preview-tts
   → models/gemini-flash-latest
   → models/gemini-2.5-flash-image
   → models/gemini-3-flash-preview
   → models/gemini-3.1-flash-image-preview
   → models/gemini-3.1-flash-tts-preview
   → models/gemini-2.5-pro
   → models/gemini-2.5-pro-preview-tts
   → models/gemini-pro-latest
   → models/gemini-3-pro-preview
   → models/gemini-3.1-pro-preview
   → models/gemini-3.1-pro-preview-customtools
   → models/gemini-3-pro-image-preview
   → models/nano-banana-pro-preview
   → models/lyria-3-pro-previ

⏳ Rate limited. Waiting 40s... (attempt 1/3)


⏳ Rate limited. Waiting 40s... (attempt 2/3)


⚠️  Quota exceeded — trying next...

⏳ Trying: models/gemini-2.0-flash-lite...


⏳ Rate limited. Waiting 40s... (attempt 1/3)


⏳ Rate limited. Waiting 40s... (attempt 2/3)


⚠️  Quota exceeded — trying next...

⏳ Trying: models/gemini-flash-lite-latest...
✅ Success with: models/gemini-flash-lite-latest

──────────────────────────────────────────────────


**🤖 Gemini (models/gemini-flash-lite-latest) says:**

Hello! How can I help you today?